# Data Preprocessing

## Import Dependencies

In [35]:
import os
import warnings

import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split

from mappers import to_numeric

warnings.filterwarnings("ignore", category=DeprecationWarning)

## Data Loading

In [36]:
root = os.path.abspath(os.path.join(os.path.dirname(__name__), "..", "data"))
path = os.path.join(root, "raw.csv")

dataset = pd.read_csv(filepath_or_buffer=path)

## Data Cleaning

#### Remove unnecessary features
1. `Category URL`

2. `Service URL`

3. `Offer URL`

4. `Offer Name`

5. `Owner URL`

6. `Owner Name`


In [37]:
unnecessary_features = ["Category URL", "Service URL", "Offer URL", "Offer Name", "Owner URL", "Owner Name"]

dataset.drop(columns=unnecessary_features, inplace=True)

#### Convert text-based values to numeric values
1. Time: `Duration`, `Offer Response Time`, and `Owner Response Time`.

2. Percentage: `Owner Completion Rate`.

3. Boolean: `Owner Verified`.

4. Money: `Price`.


In [38]:
dataset["Owner Completion Rate"] = dataset["Owner Completion Rate"].mask(dataset["Owner Completion Rate"] == "لم يحسب بعد")

dataset["Owner Completion Rate"] = dataset["Owner Completion Rate"].replace("[\%,]", "", regex=True).astype(float)

dataset["Owner Verified"] = dataset["Owner Verified"].astype(int)

dataset["Price"] = dataset["Price"].replace("[\$,]", "", regex=True).astype(float)

dataset = to_numeric(dataset, columns=["Duration", "Offer Response Time", "Owner Response Time"])

## Encoding
1. `Owner Level`

2. `Category Name`

3. `Service Name`

#### Ordinal Encoding for `Owner Level` 

In [39]:
top_prices = (dataset.groupby("Owner Level", group_keys=False).apply(lambda x: x.nlargest(1, "Price")))

top_prices["Frequency"] = top_prices.apply(lambda row: dataset[(dataset["Owner Level"] == row["Owner Level"]) & (dataset["Price"] == row["Price"])].shape[0], axis=1)

order = (top_prices[["Owner Level", "Price", "Frequency"]].sort_values(by=["Price", "Frequency"])["Owner Level"].unique())

ordinal = OrdinalEncoder(categories=[order])

dataset["Owner Level"] = ordinal.fit_transform(dataset[["Owner Level"]])

#### One-Hot Encoding for `Category Name` and `Service Name`

In [40]:
categorical_features = ["Category Name", "Service Name"]

one_hot = OneHotEncoder()

encoded = one_hot.fit_transform(dataset[categorical_features])
encoded = pd.DataFrame(encoded.toarray(), columns=[col.split("_", 1)[-1] for col in one_hot.get_feature_names_out(categorical_features)])

dataset = pd.concat([dataset, encoded], axis=1).drop(categorical_features, axis=1)

## Missing Values Handling

#### Fill missing values using KNN imputation

In [41]:
imputer = KNNImputer(n_neighbors=3)

dataset = pd.DataFrame(imputer.fit_transform(dataset), columns=dataset.columns)

## Outliers Handling

In [ ]:
# TODO: Handle outliers

##  Split the dataset into *train* and *test*

#### Using stratified random splitting for representative data and fair sampling

In [42]:
train, test = train_test_split(dataset, test_size=0.1, random_state=42, stratify=dataset["Price"])

train_path = os.path.join(root, "train.csv")
test_path = os.path.join(root, "test.csv")
 
train.to_csv(train_path, index=False)
test.to_csv(test_path, index=False)